In [ ]:
import numpy as np
from typing import Tuple
from types import SimpleNamespace
from general_functions import *
from system import *
from solvers import *

In [ ]:
my_parameters = SimpleNamespace(

    # ELECTRON PARAMETERS
    m_eff = 0.017,

    # SYSTEM PARAMETERS
    x_min = nm2au(-500.0),
    x_max = nm2au(500.0),
    y_min = nm2au(-350.0),
    y_max = nm2au(350.0),
    N_y = 100,

    # QPC ELECTRIC POTENTIAL PARAMETERS
    V_gates = eV2au(-1.0),
    sigma_x = nm2au(300.0),
    sigma_y = nm2au(300.0)

)

plot_potential_QPC(my_parameters)

In [ ]:
my_parameters = SimpleNamespace(

    # ELECTRON PARAMETERS
    m_eff = 0.017,

    # SYSTEM PARAMETERS
    x_min = nm2au(-10.0),
    x_max = nm2au(10.0),
    y_min = nm2au(-10.0),
    y_max = nm2au(10.0),
    N_y = 19,

    # QPC ELECTRIC POTENTIAL PARAMETERS
    V_gates = eV2au(0.0),
    sigma_x = nm2au(300.0),
    sigma_y = nm2au(300.0)

)

E_k, k_E = makesystem_infinite(my_parameters)

###################################################################
interesting_energies = [eV2au(0.2), eV2au(0.4)] # energies to plot the wavefunction for
n_energies_disperssion = 200 # for how many energies calculate disperssion relation in the 1st B. zone
###################################################################

# finding energy in the first Brillouin zone
dx = (my_parameters.y_max - my_parameters.y_min)/(my_parameters.N_y + 1)
my_parameters.N_x = int((my_parameters.x_max - my_parameters.x_min)/dx)
linsp_x = np.array([my_parameters.x_min + i*dx for i in range(1, my_parameters.N_x + 1)])
Es = []
for k in np.linspace(-np.pi/dx, np.pi/dx, n_energies_disperssion):
    Es.append(E_k(k))

linsp = np.linspace(-np.pi/dx, np.pi/dx, n_energies_disperssion)*nm2au(1.0)
es = np.array(Es)/eV2au(1.0)

modes = {}

fig, axs = plt.subplots(1, 2, figsize=(8, 4))

axs[0].plot(linsp, es, color='k')
axs[0].set_ylabel('$E$ (eV)'); axs[0].set_xlabel('$k_x$ (1/nm)')

axs[1].plot(linsp, es, color='k')
for energy in interesting_energies:
    mode = k_E(energy)
    modes[energy] = mode
    axs[1].plot(linsp, [energy/eV2au(1.0)]*len(linsp), '--', color = 'k', alpha = 0.3)
    axs[1].plot(np.array(mode[0])*nm2au(1.0), [energy/eV2au(1.0)]*len(mode[0]),
                'o', markersize = 3, label=f"$E$ = {energy/eV2au(1.0):.1f} eV")

axs[1].set_ylim((0, 1)); axs[1].set_xlim((-1, 1))
axs[1].set_ylabel('$E$ (eV)'); axs[1].set_xlabel('$k_x$ (1/nm)')
axs[1].legend(fancybox=False)
plt.tight_layout(); #plt.savefig("disperssion.pdf", format = "pdf");
plt.show()

linspace_y = np.array([my_parameters.y_min + i*dx for i in range(1, my_parameters.N_y + 1)])

for energy, mode in modes.items():
    ks = np.array(mode[0]); vs = np.array(mode[3])
    sorted_idxs = np.argsort(ks); ks = ks[sorted_idxs]; vs = vs[sorted_idxs]
    print(f"E = {energy/eV2au(1.0):.1f} eV"); print("k:", ks); print("v:", vs)

    fig, axs = plt.subplots(len([k for k in mode[0] if k >= 0]), 2, figsize=(8, 4*len([k for k in mode[0] if k >= 0])))
    if len([k for k in mode[0] if k >= 0]) == 1:
        axs = [axs]
    for idx, (k, u) in enumerate(zip(mode[0], mode[2])):
        if k >= 0:
            axs[idx][0].plot(linspace_y, np.real(u), color='r', label='Re')
            axs[idx][0].plot(linspace_y, np.imag(u), color='b', label='Im')
            axs[idx][0].set_ylabel(r'$\psi$ (1/$\sqrt{\text{nm}}$)'); axs[idx][0].set_xlabel(r'$x$ (nm)')
            axs[idx][0].legend(frameon=False); axs[idx][0].grid(True)
            axs[idx][0].set_title(f'k = {k*nm2au(1.0):.3} 1/nm')

            X, Y = np.meshgrid(linsp_x, linspace_y)
            Z = np.abs(u[:, np.newaxis])
            axs[idx][1].imshow(Z, extent=(my_parameters.x_min/nm2au(1.0), my_parameters.x_max/nm2au(1.0),
                                        my_parameters.y_min/nm2au(1.0), my_parameters.y_max/nm2au(1.0)),
                                        origin='lower', aspect='auto', cmap='inferno')
            axs[idx][1].set_title(r"$|\Psi|_{lead}^2$ for " + f'k = {k*nm2au(1.0):.3} 1/nm')
            axs[idx][1].set_xlabel('x (nm)'); axs[idx][1].set_ylabel('y (nm)')
    fig.tight_layout(); #plt.savefig(f"lead_psi_E{energy/eV2au(1.0):.1f}.pdf", format = "pdf");
    plt.show()

In [ ]:
Ts, Rs = [], []
Vs = np.linspace(eV2au(-1.3), eV2au(-0.7), 30)

for voltage in Vs:

    my_parameters = SimpleNamespace(

        # ELECTRON PARAMETERS
        m_eff = 0.017,

        # SYSTEM PARAMETERS
        x_min = nm2au(-500.0),
        x_max = nm2au(500.0),
        y_min = nm2au(-350.0),
        y_max = nm2au(350.0),
        N_y = 34,

        # QPC ELECTRIC POTENTIAL PARAMETERS
        V_gates = voltage,
        sigma_x = nm2au(300.0),
        sigma_y = nm2au(300.0)

    )
    
    T, R = transmission(my_parameters, np.array([eV2au(0.015)]))
    Ts.append(T[0])
    Rs.append(R[0])

fig, axs = plt.subplots(1, 3, figsize=(12, 4))

axs[0].plot(Vs/eV2au(1.0), Ts, color='r', label='T')
axs[0].set_xlabel('potential (V)')
axs[0].set_ylabel('T')
axs[0].grid()

axs[1].plot(Vs/eV2au(1.0), Rs, color='b', label='R')
axs[1].set_xlabel('potential (V)')
axs[1].set_ylabel('R')
axs[1].grid()

axs[2].plot(Vs/eV2au(1.0), np.array(Rs) + np.array(Ts), color='k', label='T + R')
axs[2].set_xlabel('potential (V)')
axs[2].set_ylabel('R + T')
axs[2].set_ylim((0, 26))
axs[2].grid()

plt.tight_layout()
#plt.savefig("TR.pdf")
plt.show()

In [ ]:
my_parameters = SimpleNamespace(

    # ELECTRON PARAMETERS
    m_eff = 0.017,

    # SYSTEM PARAMETERS
    x_min = nm2au(-500.0),
    x_max = nm2au(500.0),
    y_min = nm2au(-350.0),
    y_max = nm2au(350.0),
    N_y = 34,

    # QPC ELECTRIC POTENTIAL PARAMETERS
    V_gates = eV2au(-3e-3),
    sigma_x = nm2au(300.0),
    sigma_y = nm2au(300.0)

)

psi = makesystem_psi(pm = my_parameters, E = eV2au(0.001), plot_disperssion = True)

In [ ]:
my_parameters = SimpleNamespace(

    # ELECTRON PARAMETERS
    m_eff = 0.017,

    # SYSTEM PARAMETERS
    x_min = nm2au(-500.0),
    x_max = nm2au(500.0),
    y_min = nm2au(-350.0),
    y_max = nm2au(350.0),
    N_y = 34,

    # QPC ELECTRIC POTENTIAL PARAMETERS
    V_gates = eV2au(-1.3),
    sigma_x = nm2au(300.0),
    sigma_y = nm2au(300.0)

)

psi = makesystem_psi(pm = my_parameters, E = eV2au(0.015), plot_disperssion = True)
plot_psi_sum(my_parameters, psi)

my_parameters = SimpleNamespace(

    # ELECTRON PARAMETERS
    m_eff = 0.017,

    # SYSTEM PARAMETERS
    x_min = nm2au(-500.0),
    x_max = nm2au(500.0),
    y_min = nm2au(-350.0),
    y_max = nm2au(350.0),
    N_y = 34,

    # QPC ELECTRIC POTENTIAL PARAMETERS
    V_gates = eV2au(-1.05),
    sigma_x = nm2au(300.0),
    sigma_y = nm2au(300.0)

)

psi = makesystem_psi(pm = my_parameters, E = eV2au(0.015), plot_disperssion = True)
plot_psi_sum(my_parameters, psi)

my_parameters = SimpleNamespace(

    # ELECTRON PARAMETERS
    m_eff = 0.017,

    # SYSTEM PARAMETERS
    x_min = nm2au(-500.0),
    x_max = nm2au(500.0),
    y_min = nm2au(-350.0),
    y_max = nm2au(350.0),
    N_y = 34,

    # QPC ELECTRIC POTENTIAL PARAMETERS
    V_gates = eV2au(-.85),
    sigma_x = nm2au(300.0),
    sigma_y = nm2au(300.0)

)

psi = makesystem_psi(pm = my_parameters, E = eV2au(0.015), plot_disperssion = True)
plot_psi_sum(my_parameters, psi)